## Importing Packages

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
import mlflow.sklearn
import mlflow.xgboost

## Loading Dataset
We are using the Boston Housing dataset fetched from a public repository since scikit-learn deprecated it.

In [2]:
url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
df = pd.read_csv(url)

X = df.drop(columns=["medv"])
y = df["medv"]

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (506, 14)


,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


## Splitting Data

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

## Model Definitions

In [4]:
models = [
    (
        "Linear Regression",
        LinearRegression(),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest Regressor",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBoost Regressor",
        XGBRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        (X_train, y_train),
        (X_test, y_test)
    )
]

## Baseline Training & Evaluation

In [5]:
reports = []

for model_name, model, train_set, test_set in models:
    X_tr, y_tr = train_set
    X_te, y_te = test_set

    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)

    mae = mean_absolute_error(y_te, preds)
    mse = mean_squared_error(y_te, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_te, preds)

    report = {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2
    }
    reports.append(report)

    print("="*50)
    print(model_name)
    print("="*50)
    print(f"R2 Score: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")

Linear Regression
R2 Score: 0.7112
MAE: 3.1627
MSE: 21.5174
RMSE: 4.6387


Random Forest Regressor
R2 Score: 0.8504
MAE: 2.2751
MSE: 11.1488
RMSE: 3.3390


XGBoost Regressor
R2 Score: 0.8735
MAE: 2.1074
MSE: 9.4230
RMSE: 3.0697


## Local MLflow Experiment Tracking

In [6]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Boston Housing Regression PBLM 1")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1786002434402, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786002434402, lifecycle_stage='active', name='Boston Housing Regression PBLM 1', tags={}, trace_location=None, workspace='default'>

In [7]:
for i, (model_name, model, _, _) in enumerate(models):
    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("Model", model_name)
        if hasattr(model, "get_params"):
            mlflow.log_params(model.get_params())

        mlflow.log_metric("R2_Score", report["r2"])
        mlflow.log_metric("MAE", report["mae"])
        mlflow.log_metric("MSE", report["mse"])
        mlflow.log_metric("RMSE", report["rmse"])

        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

print("All local experiments logged successfully!")

2026/08/06 13:43:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: http://127.0.0.1:5000/#/experiments/1/runs/373ec9d270794a75ac2233c24371af6d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/08/06 13:43:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest Regressor at: http://127.0.0.1:5000/#/experiments/1/runs/c7ea96eb66e147cba71f5a2ec6bcb7e6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/08/06 13:44:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost Regressor at: http://127.0.0.1:5000/#/experiments/1/runs/6c5aee3aeecd44a1ae2058b5c46a9bc1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
All local experiments logged successfully!
